# Path 2: Order checkout flow (happy path)

Simulates a single customer's end-to-end shopping journey:

`signup -> create order -> add catalog products as line items -> read order/items -> check pending totals -> ship -> deliver -> re-check totals`

Covers `POST /customers`, `POST /products`, `POST/GET /orders`, `GET /orders/pending-totals`,
`GET /orders/:id/items`, `PATCH /orders/:id`, `POST/GET /order-items`,
`GET /customers/:id/orders`.

Run top-to-bottom (e.g. `jupyter nbconvert --to notebook --execute 02_order_checkout_flow.ipynb`).

In [ ]:
import os

import requests
from faker import Faker

BASE_URL = os.environ.get("API_BASE_URL", "http://localhost:3002/api")
fake = Faker()


def fake_product():
    return {
        "sku": f"SKU-{fake.unique.bothify(text='???-####').upper()}",
        "name": fake.unique.catch_phrase(),
        "unit_price_cents": fake.random_int(min=199, max=9999),
    }


def create_product():
    resp = requests.post(f"{BASE_URL}/products", json=fake_product())
    assert resp.status_code == 201, resp.text
    return resp.json()


def fake_order_item(order_id, product):
    return {
        "order_id": order_id,
        "product_id": product["id"],
        "quantity": fake.random_int(min=1, max=5),
        "unit_price_cents": product["unit_price_cents"],
    }


print(f"BASE_URL={BASE_URL}")

## Step 1 - A new customer signs up

In [ ]:
resp = requests.post(
    f"{BASE_URL}/customers",
    json={"email": fake.unique.email(), "password": fake.password(length=14)},
)
assert resp.status_code == 201, resp.text
customer = resp.json()
print("Customer:", customer)

## Step 2 - Customer places a new order (defaults to `pending`)

In [ ]:
resp = requests.post(f"{BASE_URL}/orders", json={"customer_id": customer["id"]})
assert resp.status_code == 201, resp.text
order = resp.json()
assert order["status"] == "pending"
print("Order:", order)

## Step 3 - Create catalog products, then add them as order line items

In [ ]:
PRODUCT_COUNT = int(os.environ.get("PRODUCT_COUNT", "4"))

products = []
order_items = []
for _ in range(PRODUCT_COUNT):
    product = create_product()
    products.append(product)
    resp = requests.post(f"{BASE_URL}/order-items", json=fake_order_item(order["id"], product))
    assert resp.status_code == 201, resp.text
    order_items.append(resp.json())

expected_total_cents = sum(i["quantity"] * i["unit_price_cents"] for i in order_items)
print(f"Added {len(order_items)} line items from {len(products)} catalog products, expected total = {expected_total_cents} cents")
order_items

## Step 4 - Read the order back, and its items via `/orders/:id/items`

In [ ]:
resp = requests.get(f"{BASE_URL}/orders/{order['id']}")
assert resp.status_code == 200, resp.text
print("Order:", resp.json())

resp = requests.get(f"{BASE_URL}/orders/{order['id']}/items")
assert resp.status_code == 200, resp.text
items = resp.json()
assert len(items) == PRODUCT_COUNT
print(f"Order has {len(items)} items")
items

## Step 5 - Check `/orders/pending-totals` includes this order with the right total

In [ ]:
resp = requests.get(f"{BASE_URL}/orders/pending-totals")
assert resp.status_code == 200, resp.text
pending_totals = resp.json()
mine = next((row for row in pending_totals if row["order_id"] == order["id"]), None)
assert mine is not None, "order missing from pending-totals"
assert int(mine["total_cents"]) == expected_total_cents, (
    f"total mismatch: expected {expected_total_cents}, got {mine['total_cents']}"
)
print("Pending total row for this order:", mine)

## Step 6 - Customer's order list shows this order

In [ ]:
resp = requests.get(f"{BASE_URL}/customers/{customer['id']}/orders")
assert resp.status_code == 200, resp.text
customer_orders = resp.json()
assert any(o["id"] == order["id"] for o in customer_orders)
print("Customer orders:", customer_orders)

## Step 7 - Warehouse ships the order (`pending -> shipped`)

In [ ]:
resp = requests.patch(f"{BASE_URL}/orders/{order['id']}", json={"status": "shipped"})
assert resp.status_code == 200, resp.text
order = resp.json()
assert order["status"] == "shipped"
print("Order after shipping:", order)

## Step 8 - Order is delivered (`shipped -> delivered`)

In [ ]:
resp = requests.patch(f"{BASE_URL}/orders/{order['id']}", json={"status": "delivered"})
assert resp.status_code == 200, resp.text
order = resp.json()
assert order["status"] == "delivered"
print("Order after delivery:", order)

## Step 9 - `/orders/pending-totals` no longer includes this order

In [ ]:
resp = requests.get(f"{BASE_URL}/orders/pending-totals")
assert resp.status_code == 200, resp.text
pending_totals = resp.json()
assert all(row["order_id"] != order["id"] for row in pending_totals), (
    "delivered order should not appear in pending-totals"
)
print("Confirmed delivered order is excluded from pending-totals")
print(f"Checkout flow complete for customer {customer['id']}, order {order['id']}")